<div style="background-color: #ffffff; color: #000000; padding: 30px;">
<img src="../media/images/kisz_logo.png" width="192" height="69" align="right" style="margin-right: 50px; margin-bottom: 50px;">
<h1>Time Series Analysis and Forecasting</h1>
</div>

<div style="background-color: #f6a800; color: #ffffff; padding: 10px;">
<h2>Solutions</h2>
<h2>Notebook C02: Machine Learning Models</h2>
</div>

Worked solutions to the 2 exercises in
[Notebook C02: Machine Learning Models](../notebooks/C02_Machine_learning_models.ipynb).

**Try each exercise yourself first.** These notebooks are most useful as a check on your reasoning, and
least useful as something to read straight through. An exercise you attempted and got wrong teaches more
than a solution you agreed with.

Where an exercise asks a question rather than requesting code, the answer is written out under the code
that produces it. Several of them have answers that are more interesting than they look.

The setup cell below reproduces the state the exercises assume, so this notebook runs on its own.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="setup">Setup</h3>
</div>

The feature matrix and the split from the notebook.

In [ ]:
import sys
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold, TimeSeriesSplit, cross_val_score

sys.path.append("../notebooks")
import nb_config

sns.set_theme(style="whitegrid")

sales = pd.read_csv(nb_config.ROSSMANN_TRAIN_PATH, parse_dates=["Date"], low_memory=False)
store = sales[sales["Store"] == 1].set_index("Date").sort_index().asfreq("D")
target = store["Sales"].astype(float)


def build_features(target, store):
    features = pd.DataFrame(index=target.index)

    for lag in (1, 2, 7, 14, 28):
        features[f"lag_{lag}"] = target.shift(lag)

    history = target.shift(1)
    for window in (7, 28):
        features[f"roll_mean_{window}"] = history.rolling(window).mean()
        features[f"roll_std_{window}"] = history.rolling(window).std()

    features["day_of_week"] = target.index.dayofweek
    features["day_of_month"] = target.index.day
    features["month"] = target.index.month
    features["days_since_start"] = (target.index - target.index[0]).days

    for k in (1, 2):
        position = target.index.dayofyear / 365.25
        features[f"fourier_sin_{k}"] = np.sin(2 * np.pi * k * position)
        features[f"fourier_cos_{k}"] = np.cos(2 * np.pi * k * position)

    features["open"] = store["Open"]
    features["promo"] = store["Promo"]
    features["school_holiday"] = store["SchoolHoliday"]

    return features


features = build_features(target, store)
complete = features.notna().all(axis=1)
X, y = features[complete], target[complete]

HOLDOUT_DAYS = 90
split = len(X) - HOLDOUT_DAYS

X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

print(f"{X.shape[1]} features, {len(X_train)} training rows, {len(X_test)} test days")

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="exercise-1">Exercise 1</h3>
</div>

> Run the same grid with `KFold(5, shuffle=True)` instead of `TimeSeriesSplit`. Does it choose the same configuration? Compare the winner's held-out MAE against the one selected above.

In [ ]:
GRID = list(itertools.product([0.03, 0.1], [7, 15, 31], [200, 600]))


def lightgbm(learning_rate, num_leaves, n_estimators):
    return lgb.LGBMRegressor(
        learning_rate=learning_rate, num_leaves=num_leaves,
        n_estimators=n_estimators, random_state=0, verbose=-1,
    )


rows = []
for learning_rate, num_leaves, n_estimators in GRID:
    settings = dict(learning_rate=learning_rate, num_leaves=num_leaves, n_estimators=n_estimators)

    scores = {
        label: -cross_val_score(
            lightgbm(**settings), X_train, y_train, cv=splitter,
            scoring="neg_mean_absolute_error",
        ).mean()
        for label, splitter in [
            ("TimeSeriesSplit", TimeSeriesSplit(5)),
            ("Shuffled", KFold(5, shuffle=True, random_state=0)),
        ]
    }

    fitted = lightgbm(**settings).fit(X_train, y_train)
    rows.append({
        **settings,
        **scores,
        "Held-out": mean_absolute_error(y_test, fitted.predict(X_test)),
    })

tuning = pd.DataFrame(rows)
tuning["ordered rank"] = tuning["TimeSeriesSplit"].rank().astype(int)
tuning["shuffled rank"] = tuning["Shuffled"].rank().astype(int)
tuning["true rank"] = tuning["Held-out"].rank().astype(int)

tuning.sort_values("Held-out").round({"TimeSeriesSplit": 1, "Shuffled": 1, "Held-out": 1})

In [ ]:
ordered_winner = tuning.loc[tuning["TimeSeriesSplit"].idxmin()]
shuffled_winner = tuning.loc[tuning["Shuffled"].idxmin()]

for label, winner in [("TimeSeriesSplit picks", ordered_winner),
                      ("Shuffled K-fold picks", shuffled_winner)]:
    print(f"{label}: learning_rate={winner['learning_rate']}, "
          f"num_leaves={int(winner['num_leaves'])}, n_estimators={int(winner['n_estimators'])}")
    print(f"    its CV estimate {winner['TimeSeriesSplit' if 'Time' in label else 'Shuffled']:6.1f}"
          f"    held-out MAE {winner['Held-out']:6.1f}")

print(f"\nBest held-out MAE anywhere in the grid: {tuning['Held-out'].min():.1f}")
print("\nRank correlation between the CV estimate and the truth:")
for label in ("TimeSeriesSplit", "Shuffled"):
    print(f"    {label:<16} {tuning[label].corr(tuning['Held-out'], method='spearman'):+.2f}")

**No, it picks a different configuration — and the choice barely matters. The estimate it reports is the
part that is wrong.**

Three separate things are going on in that table, and they are worth keeping apart.

**1. The choice differs, and costs 1.5 MAE.** `TimeSeriesSplit` picks 7 leaves, shuffled K-fold picks 15.
Held out, those score 248.2 and 249.7. Neither finds the grid's actual best, 31 leaves at 235.9, which both
rank second. On this problem the hyperparameters simply do not matter very much: all twelve
configurations land between 236 and 298, and the differences between the top few are smaller than the
noise in a 90-day evaluation.

**2. The ranking is noticeably worse.** Spearman correlation between the CV estimate and the held-out truth
is **+0.85 for the ordered split and +0.64 for the shuffled one**. Shuffled K-fold places the
second-best configuration in the grid eighth out of twelve. That ranking is what a selection procedure is
*for*, and it does it less well.

**3. The absolute level is nonsense in both, and differently nonsense.** Shuffled K-fold reports 334 where
the truth is 248; `TimeSeriesSplit` reports 458.

That last one deserves care, because it is the opposite of what the warnings usually predict. Shuffled CV
is supposed to be **optimistic** — and here it reports an error a third higher than reality.

Both numbers are off for reasons that have nothing to do with which is safer:

- `TimeSeriesSplit(5)` trains its first fold on a sixth of the data. Averaging five folds of which the
  early ones are badly under-trained gives a **pessimistic** number, not a realistic one.
- Both CV schemes evaluate across the whole training period, which includes the December trading peaks.
  The 90-day holdout is April to July, a quieter and more predictable stretch. The holdout is not a harder
  test than the CV folds; it is an easier one.

So why is the shuffled split not *more* optimistic than it is? Because the features were built correctly.
Shuffling only helps a model when neighbouring rows carry information about each other, and at this store
they barely do: consecutive open days correlate at 0.19, and everything genuinely predictable — the weekly
cycle, the closures, the promotions — is already in a column that both schemes get to see. With clean
features there is not much for a shuffled split to exploit.

**That is the honest summary, and it is more useful than "always use TimeSeriesSplit".** The damage a
shuffled split does is not a fixed penalty; it is a function of how much your rows leak into each other.
On the leaky feature set in Notebook [C01](../notebooks/C01_Feature_engineering.ipynb) it was catastrophic.
Here it costs you a slightly worse ranking and 1.5 MAE.

The reason to use the ordered split anyway is that **you cannot tell in advance which situation you are
in**. Checking would require the clean comparison above, which requires the held-out truth, which is the
thing you do not have when you are choosing. `TimeSeriesSplit` costs nothing to use and removes the
question.

One practical note on all three points: **do not read a CV score as an estimate of future error.** Use it
to rank candidates, then measure the winner on data you have not touched. Notebook
[A06](../notebooks/A06_Evaluating_models.ipynb) makes the same argument from the other direction.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="exercise-2">Exercise 2</h3>
</div>

> Apply fix 1 to the synthetic series: train the random forest on `np.diff(trending)` instead of the level, then cumulatively sum its predictions back onto the last training value. How close does it get to the linear model?

In [ ]:
rng = np.random.default_rng(0)
periods = 200
time_index = np.arange(periods)

trending = (
    100
    + 1.5 * time_index
    + 10 * np.sin(2 * np.pi * time_index / 12)
    + rng.normal(0, 5, periods)
)

trend_features = pd.DataFrame({
    "t": time_index,
    "sin": np.sin(2 * np.pi * time_index / 12),
    "cos": np.cos(2 * np.pi * time_index / 12),
})

TREND_SPLIT = 150
actual = trending[TREND_SPLIT:]


def forest():
    return RandomForestRegressor(n_estimators=200, random_state=0)


# On the level, as in the notebook
on_level = forest().fit(trend_features.iloc[:TREND_SPLIT], trending[:TREND_SPLIT])
linear = LinearRegression().fit(trend_features.iloc[:TREND_SPLIT], trending[:TREND_SPLIT])

# On the differences. np.diff shortens the series by one: differences[i] is
# trending[i + 1] - trending[i], so it pairs with the features of row i + 1.
differences = np.diff(trending)
difference_features = trend_features.iloc[1:].reset_index(drop=True)
train_end = TREND_SPLIT - 1

on_differences = forest().fit(
    difference_features.iloc[:train_end], differences[:train_end]
)

# Predicted steps, walked forward from the last value actually observed
steps = on_differences.predict(difference_features.iloc[train_end:])
reconstructed = trending[TREND_SPLIT - 1] + np.cumsum(steps)

predictions = {
    "Linear regression": linear.predict(trend_features.iloc[TREND_SPLIT:]),
    "Random forest (level)": on_level.predict(trend_features.iloc[TREND_SPLIT:]),
    "Random forest (differenced)": reconstructed,
}

for name, prediction in predictions.items():
    print(f"{name:<30} MAE {mean_absolute_error(actual, prediction):6.2f}"
          f"    final point {prediction[-1]:6.1f}")

print(f"\nActual final point {actual[-1]:.1f}; "
      f"highest value in training {trending[:TREND_SPLIT].max():.1f}")

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4.5))

ax.plot(time_index[:TREND_SPLIT], trending[:TREND_SPLIT], color="steelblue",
        linewidth=1.2, label="Train")
ax.plot(time_index[TREND_SPLIT:], actual, color="black", linewidth=1.8, label="Actual")

styles = {"Linear regression": "seagreen",
          "Random forest (level)": "crimson",
          "Random forest (differenced)": "darkorange"}
for name, prediction in predictions.items():
    ax.plot(time_index[TREND_SPLIT:], prediction, color=styles[name],
            linewidth=1.6, linestyle="--", label=name)

ax.axhline(trending[:TREND_SPLIT].max(), color="gray", linestyle=":", linewidth=1.2)
ax.text(5, trending[:TREND_SPLIT].max() + 4, "highest value seen in training",
        fontsize=9, color="gray")
ax.axvline(TREND_SPLIT, color="gray", linestyle="--", linewidth=1.0)

ax.set_title("Differencing removes the ceiling", fontsize=13, fontweight="bold")
ax.set_xlabel("Time")
ax.set_ylabel("Value")
ax.legend(loc="upper left", fontsize=9)
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

**Most of the way: MAE falls from 36.6 to 10.5, against the linear model's 3.9. Differencing closes
about 80% of the gap.**

The plot shows what changed. The random forest on levels flattens out at 331, the largest value it saw in
training, and stays there while the truth climbs to 411 — the failure the notebook demonstrates. The
differenced version has no ceiling at all: it ends at 414, slightly *above* the truth rather than 65 below
it.

The reason is that differencing changes what the model has to extrapolate. On levels, predicting month 200
means producing a number outside the range of every training label, which a tree cannot do — its output is
always an average of training labels. On differences, the target is the *step*, and the steps in the test
period look exactly like the steps in the training period: mean 1.57 in training, and the forest predicts
1.61 on average. **Nothing needs extrapolating, so there is nothing for the tree to fail at.** The trend is
rebuilt by the cumulative sum, which is arithmetic, not learning.

In [ ]:
linear_on_differences = LinearRegression().fit(
    difference_features.iloc[:train_end], differences[:train_end]
)
linear_reconstructed = trending[TREND_SPLIT - 1] + np.cumsum(
    linear_on_differences.predict(difference_features.iloc[train_end:])
)

print(f"Linear model on differences, MAE "
      f"{mean_absolute_error(actual, linear_reconstructed):.2f}")
print(f"Forest on differences, MAE      "
      f"{mean_absolute_error(actual, reconstructed):.2f}")
print()
print(f"Drift over the 50 test steps:  actual {actual[-1] - trending[TREND_SPLIT - 1]:+.1f}, "
      f"forecast {reconstructed[-1] - trending[TREND_SPLIT - 1]:+.1f}")
print(f"Mean step:  training {differences[:train_end].mean():.3f}, "
      f"predicted {steps.mean():.3f}")

**It does not close the gap entirely, and the remaining 6.6 MAE is not the forest's fault.** Fit a
*linear* model on the same differences and reconstruct it the same way, and it scores 10.2 — indistinguish-
able from the forest's 10.5, and still far behind the 3.9 that the linear model achieves on the level.

So once you difference, the choice of model stops mattering, and a new cost appears: **the cumulative sum
has no memory of the level.** Every error in a predicted step is added to the running total and never
removed. A per-step bias of 0.04 does not sound like much, but over fifty steps it compounds into an
18-unit drift, which is most of the error. The level model is anchored to reality at every point and
cannot drift; the differenced model is anchored only at the last observation and drifts freely from there.

That is the trade in one line: **differencing swaps a ceiling you cannot pass for a drift you cannot
correct.** For this series, where the trend is strong and the horizon is 50 steps, the swap is clearly
worth it — 36.6 to 10.5. For a short horizon, or a series with no trend to speak of, it can easily be the
wrong way round.

And the honest comparison at the end: the linear model on the level beats everything here, by a factor of
nearly three. The data-generating process is a straight line plus a sine wave plus noise, and
`LinearRegression` on `t`, `sin` and `cos` is that process exactly. No amount of repair to the wrong model
class catches a model that has the right functional form. Fix 1 makes the forest usable on a trending
series; it does not make it the right choice for one.

The third option, which Notebook [C03](../notebooks/C03_Ensembles.ipynb) picks up, is to stop choosing:
let the linear model carry the trend and let the trees model what is left over.

---

Back to [Notebook C02](../notebooks/C02_Machine_learning_models.ipynb), or on to
[Notebook C03](../notebooks/C03_Ensembles.ipynb).